## Example 1: zope.interface functionality test 1

In [ ]:
from zope.interface import Interface, implementer, Attribute

# Example 1: Defining and implementing interfaces
print("Example 1: Interface definition and implementation")

class IGreeter(Interface):
    """Interface for greeting objects"""
    name = Attribute("The name of the greeter")
    
    def greet(person):
        """Greet a person"""

@implementer(IGreeter)
class EnglishGreeter:
    def __init__(self, name):
        self.name = name
    
    def greet(self, person):
        return f"Hello {person}, I'm {self.name}"

greeter = EnglishGreeter("Alice")
print(f"Greeter name: {greeter.name}")
print(f"Greeting: {greeter.greet('Bob')}")
print(f"Implements IGreeter: {IGreeter.providedBy(greeter)}")
print("Example 1 completed")

## Example 2: zope.interface functionality test 2

In [ ]:
from zope.interface import Interface, implementer
from zope.interface.adapter import AdapterRegistry

# Example 2: Adapter pattern with interface adaptation
print("Example 2: Adapter pattern")

class IAnimal(Interface):
    def make_sound():
        """Make a sound"""

@implementer(IAnimal)
class Dog:
    def make_sound(self):
        return "Woof!"

# Adapt to string representation
def animal_to_string(animal):
    return f"Animal says: {animal.make_sound()}"

registry = AdapterRegistry()
registry.register([IAnimal], Interface, '', animal_to_string)

dog = Dog()
print(f"Dog sound: {dog.make_sound()}")
adapted = registry.lookup([IAnimal], Interface, '')
print(f"Adapted: {adapted(dog)}")
print("Example 2 completed")

## Example 3: zope.interface functionality test 3

In [ ]:
from zope.interface import Interface, implementer, invariant, Invalid, Attribute

# Example 3: Interface invariants for validation
print("Example 3: Interface invariants")

def valid_age(obj):
    if obj.age < 0 or obj.age > 150:
        raise Invalid("Age must be between 0 and 150")

class IPerson(Interface):
    age = Attribute("Person's age")
    invariant(valid_age)

@implementer(IPerson)
class Person:
    def __init__(self, age):
        self.age = age

# Valid person
person1 = Person(30)
print(f"Person 1 age: {person1.age}")
print(f"Implements IPerson: {IPerson.providedBy(person1)}")
assert IPerson.providedBy(person1)

# verifyObject checks the declared attributes; it raises Invalid (BrokenImplementation) on a regression and never
# runs invariants (measured: Person(200) verifies True). validateInvariants() is what runs valid_age.
from zope.interface.verify import verifyObject
assert verifyObject(IPerson, person1) is True
IPerson.validateInvariants(person1)
print("Person 1 validation: PASSED")

person2 = Person(200)
try:
    IPerson.validateInvariants(person2)
except Invalid as e:
    assert str(e) == "Age must be between 0 and 150", e
    print(f"Person 2 (age 200) rejected as expected: {e}")
else:
    raise AssertionError("validateInvariants() accepted age 200")

print("Example 3 completed")

## Example 4: zope.interface functionality test 4

In [ ]:
from zope.interface import Interface, implementer, classImplements

# Example 4: Runtime interface implementation
print("Example 4: Runtime interface implementation")

class ISerializable(Interface):
    def to_dict():
        """Convert to dictionary"""

class Product:
    def __init__(self, name, price):
        self.name = name
        self.price = price

# Add interface implementation at runtime
classImplements(Product, ISerializable)

def to_dict(self):
    return {"name": self.name, "price": self.price}

Product.to_dict = to_dict

product = Product("Laptop", 999.99)
print(f"Product: {product.name} - ${product.price}")
print(f"Implements ISerializable: {ISerializable.providedBy(product)}")
print(f"Serialized: {product.to_dict()}")
print("Example 4 completed")

## Example 5: zope.interface functionality test 5

In [ ]:
from zope.interface import Interface, implementer, directlyProvides
from zope.interface.verify import verifyClass, verifyObject

# Example 5: Interface verification
print("Example 5: Interface verification")

class ICache(Interface):
    def get(key):
        """Get value by key"""
    def set(key, value):
        """Set value for key"""

@implementer(ICache)
class MemoryCache:
    def __init__(self):
        self.data = {}
    
    def get(self, key):
        return self.data.get(key)
    
    def set(self, key, value):
        self.data[key] = value

# Verify class implements interface
verifyClass(ICache, MemoryCache)
print("Class verification: PASSED")

# Verify instance provides interface
cache = MemoryCache()
verifyObject(ICache, cache)
print("Instance verification: PASSED")

cache.set("user:1", "Alice")
print(f"Cached value: {cache.get('user:1')}")
print("Example 5 completed")